# PPE Detection Project - Google Colab

Notebook này giúp chạy project trên Google Colab: cài dependency, chuẩn bị dataset Kaggle/Roboflow, train YOLO, evaluate và detect ảnh mẫu.

**Trước khi chạy:** vào `Runtime → Change runtime type → GPU` nếu muốn train nhanh hơn.

## 1. Chuẩn bị project

Có 2 cách:

1. Nếu project đã được đưa lên GitHub, điền `REPO_URL` rồi chạy cell.
2. Nếu bạn upload/copy sẵn thư mục `ppe_detection_project` vào `/content`, để `REPO_URL = ""` và chạy tiếp.

In [ ]:
from pathlib import Path
import os
import zipfile

REPO_URL = ""  # Ví dụ: "https://github.com/<user>/<repo>.git"
PROJECT_DIR = Path("/content/ppe_detection_project")

if not PROJECT_DIR.exists():
    if REPO_URL:
        !git clone {REPO_URL} /content/repo
        candidates = list(Path("/content/repo").rglob("ppe_detection_project"))
        if not candidates:
            raise FileNotFoundError("Không tìm thấy thư mục ppe_detection_project trong repo vừa clone.")
        PROJECT_DIR = candidates[0]
    else:
        from google.colab import files
        print("Chưa có /content/ppe_detection_project. Upload file zip chứa thư mục ppe_detection_project...")
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if not zip_names:
            raise FileNotFoundError("Hãy upload file .zip chứa thư mục ppe_detection_project hoặc điền REPO_URL.")
        extract_root = Path("/content/uploaded_project")
        extract_root.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_names[0]) as archive:
            archive.extractall(extract_root)
        candidates = list(extract_root.rglob("ppe_detection_project"))
        if not candidates:
            raise FileNotFoundError("File zip không chứa thư mục ppe_detection_project.")
        PROJECT_DIR = candidates[0]

os.chdir(PROJECT_DIR)
print("Project dir:", PROJECT_DIR)
!pwd
!find . -maxdepth 2 -type f | sort | head -50

Chưa có /content/ppe_detection_project. Upload file zip chứa thư mục ppe_detection_project...


## 2. Cài đặt dependencies

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Cấu hình Kaggle và tải dataset

Dataset dùng trong project: `snehilsanyal/construction-site-safety-image-dataset-roboflow`.

Nếu chưa có credential Kaggle:
1. Vào Kaggle → Account → Create New API Token để tải `kaggle.json`.
2. Chạy cell dưới và upload file `kaggle.json`.

In [ ]:
from pathlib import Path
from google.colab import files
import os

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Upload kaggle.json...")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Bạn cần upload đúng file kaggle.json")
    (Path("kaggle.json")).replace(kaggle_json)

os.chmod(kaggle_json, 0o600)
print("Kaggle credential ready:", kaggle_json)

In [ ]:
!python src/prepare_css_dataset.py   --download   --output /content/datasets/construction_site_safety   --yaml-output data/data.yaml   --copy   --force

!cat data/data.yaml

## 4. Train YOLO

Có thể đổi `MODEL`, `EPOCHS`, `IMGSZ`, `BATCH`. Colab GPU thường dùng `DEVICE = 0`; nếu không có GPU thì đặt `DEVICE = "cpu"`.

In [ ]:
MODEL = "yolov8n.pt"   # hoặc "yolo11n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
DEVICE = 0 if torch.cuda.is_available() else "cpu"

!python src/train.py   --data data/data.yaml   --model {MODEL}   --epochs {EPOCHS}   --imgsz {IMGSZ}   --batch {BATCH}   --device {DEVICE}

## 5. Evaluate model

In [ ]:
BEST_MODEL = "runs/train/ppe_yolo/weights/best.pt"

!python src/evaluate.py   --model {BEST_MODEL}   --data data/data.yaml   --split val   --device {DEVICE}

## 6. Detect ảnh upload trong Colab

In [ ]:
from google.colab import files
from IPython.display import Image as IPImage, display
from pathlib import Path

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError("Chưa upload ảnh.")

image_path = Path(next(iter(uploaded.keys()))).resolve()
!python src/detect_image.py   --model {BEST_MODEL}   --source {image_path}   --output runs/detect/colab_image   --conf 0.25

output_path = Path("runs/detect/colab_image") / f"{image_path.stem}_detected{image_path.suffix}"
display(IPImage(filename=str(output_path)))

## 7. Lưu kết quả

Có thể tải trực tiếp file zip hoặc copy sang Google Drive.

In [ ]:
!zip -r /content/ppe_runs.zip runs data/data.yaml
files.download('/content/ppe_runs.zip')